# Anima: booru counterpoint pairs (which tags steer the stock model, and what else they move)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/anima-trainer/blob/main/notebooks/anima_colab_counterpoint.ipynb)

**[Anima](https://huggingface.co/circlestone-labs/Anima)** learned from pictures described by Danbooru tags, so its own
tags are the natural handles for steering it. A *counterpoint pair* is two tags from one slot of that vocabulary (two hair
colours, two eye colours, two body sizes), so a picture normally carries one of them. Before a pair can carry a slider,
the stock model has to show two things: each tag moves the picture its own way, and it moves only its own attribute.

This notebook draws each tag on eight adult characters x two seeds, scores every picture with an anime image tagger
(WD EVA02-Large v3), and uploads the result into `experiments/e050_anima_counterpoint_words` of
[AbstractPhil/geolip-beatrix-anima](https://huggingface.co/AbstractPhil/geolip-beatrix-anima).

| pair | + tag | - tag | framing |
|---|---|---|---|
| hair colour | blonde hair | black hair | upper body |
| eye colour | red eyes | blue eyes | upper body |
| body size | plump | skinny | full body |
| expression | smile | frown | upper body |
| eye openness | wide-eyed | half-closed eyes | upper body |

More pairs are ready to switch on in step 3: hair length, eye shape (tareme / tsurime), figure scale (giant / mini
person), a second eye-colour and hair-colour pair, monochrome, mature female, chibi.

**Nothing is drawn twice.** Every set (one tag at one framing over the 16 cells) is saved to
[AbstractPhil/geolip-beatrix-anima-data](https://huggingface.co/datasets/AbstractPhil/geolip-beatrix-anima-data)
under `sets/cp_*` as soon as it is drawn. A rerun, a fresh runtime or a later experiment with the same settings downloads
it instead of drawing it again, and the run prints how many pictures came from there and how many it drew. Each picture
is stored with the plain prompt as its caption, so the sets can train sliders later.

**Content screen.** The tagger also rates every picture. A picture rated questionable or explicit is left off the
contact sheets, and its set is not uploaded.

**The rule (fixed before the run).** For each tag, over the 16 cells: the tag *moves it* when the tagger's score moves
the expected way in at least 12 of the 16 cells and the mean is beyond 3 standard errors. A pair is *a handle* when both
of its tags move it. A pair *overlaps* another when its tags move that pair's score by more than a third of that pair's
own range. A handle with no overlap goes on to the slider stage.

**Cost.** 12 sets x 16 = 192 pictures at 768 px and 30 steps on the RTX PRO 6000 (the run prints time spent and time
left), plus the setup: the dependencies, the three model files (~5.6 GB) and the tagger (1.3 GB).

## 0 · Confirm the GPU and disk

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /content | tail -1

## 1 · Bootstrap: the runner at its release, and the dependencies (restart ONCE the first time)

Clones the runner at the release this notebook was written for, and the diffusion-pipe fork beside it (its Anima model
code draws the pictures), then installs the dependencies. **Run the cell; if the kernel restarts, run it again** (it
then skips the install).

In [ ]:
import os, sys, subprocess
TAG = "v0.2.0"                       # this notebook's release of the runner
REPO = os.environ.get("ANIMA_REPO", "/content/anima-trainer")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--quiet", "https://github.com/AbstractEyes/anima-trainer.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "--quiet", "--tags", "origin"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "--quiet", TAG], check=True)
sys.path.insert(0, REPO)
import anima_colab
if anima_colab.install(REPO, dp_url=anima_colab.DP_FORK_URL, similarity=False):   # + the diffusion-pipe fork
    os.kill(os.getpid(), 9)             # -> restart; reconnect and RE-RUN this cell (then it skips)

## 2 · Set up

Checks the GPU, finds the fork and downloads the three model files. Needs a **write** `HF_TOKEN` in Colab Secrets: the
results and the drawn sets are uploaded.

In [ ]:
from geolip_anima_trainer.anima_runner import AnimaRunner
a = AnimaRunner()      # Anima-Base v1.0 at 768 px; results -> AbstractPhil/geolip-beatrix-anima, sets -> ...-anima-data
a.setup()              # env + HF login + GPU check + the fork + the three model files (~5.6 GB)

## 3 · The counterpoint pairs

Prints the job's size first (how many pictures come from the data repo and how many it draws), then time spent and time
left, one line per set with its own score against the plain prompt, and at the end one line per pair: each tag against
the plain prompt, the verdict, and any overlap. To add pairs, extend the list; sets already drawn come back from the
data repo, and a run that is already done for the same list is skipped.

In [ ]:
from geolip_anima_trainer import anima_experiments as ax
PAIRS = list(ax.CP_DEFAULT)      # hair colour, eye colour, body size, expression, eye openness
# PAIRS += ["hair_length", "eye_shape", "figure_scale"]     # more pairs: ax.CP_PAIRS lists them all
meta = a.run_counterpoint_words(PAIRS)

## Notes

- **Where things land:** `experiments/e050_anima_counterpoint_words/` in the experiments repo (README with the question,
  the design, the rule fixed before the run and the result; `meta.json`; `result.json`; `tagger_probs.npz` with every
  picture's probability for each of the tagger's tags, so another tag can be read without the pictures; one contact
  sheet per pair). The sets: `sets/cp_<tag>_<framing>-<key>/` in the data repo. Locally: `/content/anima_data/`.
- **After a disconnect:** run steps 1-3 again. Finished sets come back from the data repo; only the unfinished set is
  drawn again. `a.save_training_sets()` uploads any complete set still on this runtime (a set the content screen held
  back stays local).
- **The pictures** are outputs of Anima-Base v1.0 (CircleStone Labs Non-Commercial License: outputs are not derivatives
  of the model; the restriction applies to the model and its derivatives, such as LoRAs).